# `safim_eval_1000` — Phase B: Base vs Random-LoRA vs Distributed-LoRA (GPU + W&B)
**Kaggle notebook — thin controller only. All logic lives in `scripts/run_safim_eval_1000.py` and `src/training/evaluate.py`.**

This is the **second** of the two `safim_eval_1000` notebooks. It consumes the fixed eval set built by `safim_eval_1000_dataset.ipynb` and runs the three-way comparison.

`scripts/run_safim_eval_1000.py`, driven by `configs/data/safim_eval_1000.yaml`, evaluates **Base (`Qwen/Qwen2.5-Coder-0.5B`)**, **`experiment/random_model/seed42`**, and **`experiment/distributed_model/seed42`** on the *same* 1000 fixed FIM tasks — `exact_match` / `edit_similarity`, aggregate and broken down by `fim_type` bucket.

**Resumable** — each model's progress is checkpointed to a local JSONL and mirrored to `experiment/safim_eval_1000_results/` in the HF dataset repo every 50 records; re-running after a crash or session restart picks up mid-model instead of starting over.

**Monitoring (W&B)** — when `WANDB_API_KEY` is set, the run logs to the `stack-v3-python-fim-data` project (`job_type="evaluation"`, `group="experiment-42"`, so it lands on the same comparison panel as the pilot's training / FIM-generation runs). Logged: per-model aggregate `exact_match` / `edit_similarity`, a by-bucket `wandb.Table`, and every plot as a `wandb.Image` (`comparison.png`, `exact_match_by_bucket.png`, `edit_similarity_by_bucket.png`) plus a `safim-eval-1000-plots` artifact bundling the PNGs + CSVs. Without the key the eval still runs and writes the same graphs locally.

> **Prerequisite:** run `safim_eval_1000_dataset.ipynb` first — this notebook downloads `experiment/safim_eval_1000/` and fails if it does not exist yet.

> **GPU required** — 3 models × up to 1000 unbatched greedy-decode generations. Set the accelerator to a GPU before running Cell 5. For a fast first-time calibration, run Cell 5 once with `--max-samples 50`, note the printed s/sample + ETA, then re-run without it for the full 1000.

In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", REPO_DIR, "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install dependencies (GPU — model loading + inference + W&B) ──
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "unsloth", "peft", "trl", "transformers", "torch",
     "pandas", "seaborn", "matplotlib", "wandb", "weave"],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face (+ optional W&B) ──────────
# HF_TOKEN / WANDB_API_KEY are stored as Kaggle Secrets — NEVER hardcode.
# Add them: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# W&B tracking is optional — scripts/run_safim_eval_1000.py's
# init_wandb_run() prints a warning and skips W&B logging if this
# secret isn't set (the eval still runs, graphs stay local).
try:
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✓ HF_TOKEN and WANDB_API_KEY loaded — comparison graphs will log to W&B")
except Exception:
    print("✓ HF_TOKEN loaded. WANDB_API_KEY not set — W&B logging will be skipped.")


In [ ]:
# ── Cell 4: GPU check (Phase B needs one) ──────────────────
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU available — ready to run the 3-way eval below.")
    print(result.stdout.split("\n")[8] if len(result.stdout.split("\n")) > 8 else "")
else:
    raise SystemExit(
        "✗ No GPU on this session. Switch the accelerator to a GPU "
        "(Settings → Accelerator), restart, and re-run cells 1–4 before continuing."
    )


## Run the three-way comparison (resumable, logs graphs to W&B)

In [ ]:
# ── Cell 5: Evaluate Base / Random-LoRA / Distributed-LoRA on the fixed 1000 ──
# Resumable: if this crashes or the session restarts partway through,
# just re-run this cell — each model's already-scored tasks (local +
# HF-mirrored checkpoint) are skipped, not re-generated.
#
# Fast first-time calibration: append  --max-samples 50  once, note the
# printed s/sample + ETA, then re-run without it for the full 1000.
import subprocess
import sys

os.chdir(REPO_DIR)

subprocess.run(
    [sys.executable, "scripts/run_safim_eval_1000.py",
     "--config", "configs/data/safim_eval_1000.yaml"],
    check=True,
)


In [ ]:
# ── Cell 6: Display results ──────────────────────────
import pandas as pd
from IPython.display import Image, display

output_dir = "results/safim_eval_1000"

print("Aggregate (Base vs Random-LoRA vs Distributed-LoRA):")
display(pd.read_csv(f"{output_dir}/metrics.csv"))

print("\nBy FIM bucket:")
display(pd.read_csv(f"{output_dir}/metrics_by_bucket.csv"))

for plot in ["comparison.png", "exact_match_by_bucket.png", "edit_similarity_by_bucket.png"]:
    display(Image(filename=f"{output_dir}/plots/{plot}"))

print("\n" + "=" * 70)
print("If WANDB_API_KEY was set, the same tables + plots are on the W&B\n"
      "run printed above (project: stack-v3-python-fim-data, "
      "job_type=evaluation, group=experiment-42).")
